# Exploitation Zone — Knowledge Graph ABOX construction

**Projecte 2 — BDA-GIA (UPC)**

Aquest notebook crea instancies al KG a partir de les taules que surten de la Trusted Zone i la metadata (TBOX.owl), utilitzant  RDFLib Graph.


## 0. Instal·lació de dependències

In [ ]:
#!pip install rdflib duckdb pandas

## 1. Imports i configuració

In [ ]:
import duckdb
import pandas as pd
import rdflib
from rdflib import Graph, Namespace, URIRef, Literal, RDF, RDFS, OWL
from rdflib.namespace import XSD
import urllib.parse
import os

# ──────────────────────────────────────────────
# CONFIGURACIÓ DE PATHS 

DUCKDB_PATH = "../trust_zone/trusted_zone.db"   # ruta a la Trusted Zone
TBOX_PATH   = "TBOX.owl"                         # TBOX al mateix directori
OUTPUT_KG   = "kg_accidentes_bcn.ttl"            # fitxer de sortida del KG

## 2. Definició de Namespaces

Tots els URIs del TBOX provenen de WebProtégé. Definim el namespace base i les propietats/classes tal com estan al fitxer `TBOX.owl`.

In [4]:
# Namespace base del TBOX (WebProtégé)
WP = Namespace("http://webprotege.stanford.edu/")

# Namespace per a les instàncies (ABOX)
DATA = Namespace("http://data.bcn.cat/accidentes/")

# ── Classes ──────────────────────────────────────────────────────────
C_ACCIDENTE           = WP["RDHGr7pl4B25glJmq2xYYmN"]
C_ACCIDENTE_LEVE      = WP["R7oZUszqj7aEjlsk698JmFM"]
C_ACCIDENTE_GRAVE     = WP["RYqhWnA3Yc3qHHEFBZcRoJ"]
C_ACCIDENTE_MORTAL    = WP["R8bzivJwhQYj45eHewQTg08"]
C_ACCIDENTE_SIN_VIC   = WP["RCyHtlgkOu2iks7ZQbQPeLw"]
C_LUGAR               = WP["RCglr2LgVmFa85WTIfryLqR"]
C_PERSONA             = WP["R8CyWdnPgD3YkoaC4jXyLH8"]
C_CONDUCTOR           = WP["RBPXhVTcni9UaQtHolrwEGM"]
C_PASSATGER           = WP["R9NI0gPxFQhpaSZpsxWyRWC"]
C_VIANANT             = WP["RBjMNvYweizmQ8EE1HnWwNF"]
C_ESTACIO_METEO       = WP["RDfBEkaNQRJf9P95RTve7DV"]
C_LECTURA_METEO       = WP["R8I54xpcV9jUWT2m34cWCxd"]

# ── Object Properties ────────────────────────────────────────────────
P_UBICADO_EN          = WP["Ro80knf6kx29DunELcteYN"]   # Accidente/Estacio → Lugar
P_INVOLUCRADO_EN      = WP["Rqp27L1N9oTZweWd4AbJKw"]   # Persona → Accidente
P_MEDIR               = WP["RDc1x0FCFi3bcbVBIr3mn8a"]   # Estacio → LecturaMeteo

# ── Data Properties ──────────────────────────────────────────────────
P_ACCIDENT_ID         = WP["RiUMyEG1nwNphv16JMNmsZ"]
P_FECHADO_EN          = WP["R8BfMxHvuSS21BCIoJe4qQ8"]
P_NUM_MORT            = WP["RDLF2iVJgjiRFE4TYcqZ6a3"]
P_NUM_FERIT_LLEU      = WP["R8RNUHdxy4iNBMh6DiKMNHR"]
P_NUM_FERIT_GREU      = WP["R9KzZ5w47b2GzTS1wFlEzB6"]
P_CALLE               = WP["R9J0LyEyfULTAx5DDRGG4Ys"]
P_BARRIO              = WP["RCJq0kcAKOAezhCOBeEqGlT"]
P_DISTRITO            = WP["R8VCcKl4uCPbHDX8to2A5w8"]
P_EDAD                = WP["RCsWF1rtGUwHlrCw2nFCiyE"]
P_SEXO                = WP["RDEVg1ttXmeqGODajUtvvgq"]
P_MOTIU_DESPL         = WP["RB1tANRENUq7XbDeRnBbE3J"]
P_VEHICLE_IMPLICAT    = WP["R94iV1JYRs5IqQSGPTpJbM8"]
P_ESTACIO_ID          = WP["R9DzTUzW9I8YOWAWyrQbYXj"]
P_ACRONIMO            = WP["R9REQUrspa6BUM46zxZhfsg"]
P_VALOR               = WP["R7sjUhrXMjtfE85znEWI95G"]

## 3. Lectura de la Trusted Zone (DuckDB)

In [5]:
conn = duckdb.connect(DUCKDB_PATH, read_only=True)

df_accidents = conn.execute("SELECT * FROM T_ACCIDENTS").df()
df_persones  = conn.execute("SELECT * FROM T_PERSONES").df()
df_meteo     = conn.execute("SELECT * FROM T_METEO").df()

conn.close()

print(f"T_ACCIDENTS: {len(df_accidents):,} files")
print(f"T_PERSONES:  {len(df_persones):,} files")
print(f"T_METEO:     {len(df_meteo):,} files")
df_accidents.head(3)

T_ACCIDENTS: 7,741 files
T_PERSONES:  16,001 files
T_METEO:     16,396 files


,numero_expedient,codi_districte,nom_districte,codi_barri,nom_barri,codi_carrer,nom_carrer,num_postal,descripcio_dia_setmana,nk_any,...,descripcio_causa_vianant,numero_morts,numero_lesionats_lleus,numero_lesionats_greus,numero_victimes,numero_vehicles_implicats,coordenada_utm_y_ed50,coordenada_utm_x_ed50,longitud_wgs84,latitud_wgs84
0,2025S000175,10,Sant Martí,72,Sant Martí de Provençals,169409,Corts Catalanes,989-991,Diumenge,2025,...,Desconegut,0,0,0,0,1,433080.792,4585086.167,2.198167,41.412694
1,2025S000233,2,Eixample,6,la Sagrada Família,350308,València,Desconegut,Dimarts,2025,...,Desconegut,0,0,0,0,2,431074.797,4583801.691,2.174313,41.400955
2,2025S000273,7,Horta-Guinardó,35,el Guinardó,365702,Mare de Déu de Montserrat,89-103,Dimecres,2025,...,Altres,0,1,0,1,1,430529.797,4585444.552,2.167605,41.415705


## 4. Construcció del Graf RDF (ABOX)

Creem el graf carregant primer el TBOX i després afegint les instàncies (ABOX) a partir de les taules de DuckDB.

### Funció auxiliar: classificar el tipus d'accident

In [ ]:
def accident_class(row):
    """Retorna la classe OWL correcta segons el nombre de víctimes."""
    morts = row.get("numero_morts", 0) or 0 # or 0 per assegurar que no sigui None
    greus = row.get("numero_lesionats_greus", 0) or 0
    lleus = row.get("numero_lesionats_lleus", 0) or 0
    victimes = row.get("numero_victimes", 0) or 0

    if morts > 0:
        return C_ACCIDENTE_MORTAL
    elif greus > 0:
        return C_ACCIDENTE_GRAVE
    elif lleus > 0:
        return C_ACCIDENTE_LEVE
    else:
        return C_ACCIDENTE_SIN_VIC


def person_class(descripcio):
    """Retorna la subclasse de Persona correcta."""
    if not descripcio:
        return C_PERSONA
    d = str(descripcio).lower()

    if "Conductor" in d:
        return C_CONDUCTOR
    elif "Passatger" in d:
        return C_PASSATGER
    elif "Vianant" in d:
        return C_VIANANT
    return C_PERSONA


def safe_uri(text):
    """Codifica un text per ser usat com a fragment d'URI."""
    return urllib.parse.quote(str(text).strip().replace(" ", "_"), safe="")



### 4.1 Inicialitzar el graf i carregar el TBOX

In [7]:
g = Graph()

# Carregar el TBOX (schema/metamodel)
g.parse(TBOX_PATH)
print(f"TBOX carregat: {len(g)} triples de schema")

# Lligar namespaces per a queries llegibles
g.bind("wp",   WP)
g.bind("data", DATA)
g.bind("rdf",  RDF)
g.bind("rdfs", RDFS)
g.bind("xsd",  XSD)

TBOX carregat: 129 triples de schema


### 4.2 Poblar accidents i llocs

In [8]:
# Conjunt per evitar duplicats de Llocs
llocs_vistos = set()

for _, row in df_accidents.iterrows():
    exp_id = str(row["numero_expedient"]).strip()

    # ── URI de l'accident ──────────────────────────────────────────────
    acc_uri = DATA[f"accident/{safe_uri(exp_id)}"]

    # Tipus (subclasse d'Accidente)
    g.add((acc_uri, RDF.type, accident_class(row)))
    g.add((acc_uri, RDF.type, C_ACCIDENTE))  # classe general també

    # Propietats de l'accident
    g.add((acc_uri, P_ACCIDENT_ID, Literal(exp_id, datatype=XSD.string)))

    if pd.notna(row.get("numero_morts")):
        g.add((acc_uri, P_NUM_MORT,       Literal(int(row["numero_morts"]),            datatype=XSD.integer)))
    if pd.notna(row.get("numero_lesionats_lleus")):
        g.add((acc_uri, P_NUM_FERIT_LLEU, Literal(int(row["numero_lesionats_lleus"]), datatype=XSD.integer)))
    if pd.notna(row.get("numero_lesionats_greus")):
        g.add((acc_uri, P_NUM_FERIT_GREU, Literal(int(row["numero_lesionats_greus"]), datatype=XSD.integer)))

    # Data i hora com a string ISO (dateTimeStamp requereix timezone)
    any_ = row.get("nk_any", "")
    mes  = row.get("mes_any", "")
    dia  = row.get("dia_mes", "")
    hora = row.get("hora_dia", 0) or 0
    if pd.notna(any_) and pd.notna(mes) and pd.notna(dia):
        dt_str = f"{int(any_):04d}-{int(mes):02d}-{int(dia):02d}T{int(hora):02d}:00:00"
        g.add((acc_uri, P_FECHADO_EN, Literal(dt_str, datatype=XSD.string)))

    # ── URI del Lloc ───────────────────────────────────────────────────
    codi_carrer  = row.get("codi_carrer", "")
    num_postal   = row.get("num_postal", "")
    lloc_key     = f"{codi_carrer}_{num_postal}"
    lloc_uri     = DATA[f"lloc/{safe_uri(lloc_key)}"]

    if lloc_key not in llocs_vistos:
        g.add((lloc_uri, RDF.type, C_LUGAR))
        if pd.notna(row.get("nom_carrer")):
            g.add((lloc_uri, P_CALLE,    Literal(str(row["nom_carrer"]), datatype=XSD.string)))
        if pd.notna(row.get("nom_barri")):
            g.add((lloc_uri, P_BARRIO,   Literal(str(row["nom_barri"]),  datatype=XSD.string)))
        if pd.notna(row.get("nom_districte")):
            g.add((lloc_uri, P_DISTRITO, Literal(str(row["nom_districte"]), datatype=XSD.string)))
        llocs_vistos.add(lloc_key)

    # Relació Accident → Lloc
    g.add((acc_uri, P_UBICADO_EN, lloc_uri))

print(f"✓ Accidents afegits. Total triples ara: {len(g):,}")

✓ Accidents afegits. Total triples ara: 79,601


### 4.3 Poblar persones

In [9]:
for i, row in df_persones.iterrows():
    exp_id   = str(row["numero_expedient"]).strip()
    pers_uri = DATA[f"persona/{safe_uri(exp_id)}_{i}"]
    acc_uri  = DATA[f"accident/{safe_uri(exp_id)}"]

    # Tipus de persona
    g.add((pers_uri, RDF.type, person_class(row.get("descripcio_tipus_persona"))))

    if pd.notna(row.get("edat")):
        g.add((pers_uri, P_EDAD, Literal(int(row["edat"]), datatype=XSD.integer)))
    if pd.notna(row.get("descripcio_sexe")) and str(row["descripcio_sexe"]).strip():
        g.add((pers_uri, P_SEXO, Literal(str(row["descripcio_sexe"]), datatype=XSD.string)))
    if pd.notna(row.get("desc_tipus_vehicle_implicat")) and str(row["desc_tipus_vehicle_implicat"]).strip():
        g.add((pers_uri, P_VEHICLE_IMPLICAT, Literal(str(row["desc_tipus_vehicle_implicat"]), datatype=XSD.string)))

    motiu = row.get("descripcio_motiu_desplacament_conductor") or row.get("descripcio_motiu_desplacament_vianant")
    if pd.notna(motiu) and str(motiu).strip():
        g.add((pers_uri, P_MOTIU_DESPL, Literal(str(motiu), datatype=XSD.string)))

    # Relació Persona → Accident
    g.add((pers_uri, P_INVOLUCRADO_EN, acc_uri))

print(f"✓ Persones afegides. Total triples ara: {len(g):,}")

✓ Persones afegides. Total triples ara: 175,607


### 4.4 Poblar estacions meteorològiques i lectures

In [10]:
estacions_vistes = set()

for i, row in df_meteo.iterrows():
    codi_est  = str(row["codi_estacio"]).strip()
    est_uri   = DATA[f"estacio/{safe_uri(codi_est)}"]

    # Crear estació (una sola vegada)
    if codi_est not in estacions_vistes:
        g.add((est_uri, RDF.type,    C_ESTACIO_METEO))
        g.add((est_uri, P_ESTACIO_ID, Literal(codi_est, datatype=XSD.string)))
        estacions_vistes.add(codi_est)

    # Cada fila és una LecturaMeteorologica
    lect_uri = DATA[f"lectura/{safe_uri(codi_est)}_{i}"]
    g.add((lect_uri, RDF.type,   C_LECTURA_METEO))

    if pd.notna(row.get("acronim")):
        g.add((lect_uri, P_ACRONIMO,   Literal(str(row["acronim"]),  datatype=XSD.string)))
    if pd.notna(row.get("valor")):
        g.add((lect_uri, P_VALOR,      Literal(float(row["valor"]), datatype=XSD.float)))
    if pd.notna(row.get("data_lectura")):
        g.add((lect_uri, P_FECHADO_EN, Literal(str(row["data_lectura"]), datatype=XSD.string)))

    # Relació Estació → Lectura
    g.add((est_uri, P_MEDIR, lect_uri))

print(f"✓ Dades meteorològiques afegides. Total triples: {len(g):,}")

✓ Dades meteorològiques afegides. Total triples: 257,593


### 4.5 Serialitzar el KG a disc

In [11]:
g.serialize(OUTPUT_KG, format="turtle")
size_mb = os.path.getsize(OUTPUT_KG) / 1_000_000
print(f"✓ KG serialitzat a '{OUTPUT_KG}' ({size_mb:.2f} MB, {len(g):,} triples)")

✓ KG serialitzat a 'kg_accidentes_bcn.ttl' (15.71 MB, 257,593 triples)


In [13]:
def run_sparql(query_str, title=""):
    """Executa una query SPARQL sobre el graf i retorna un DataFrame."""
    results = g.query(query_str)
    cols = [str(v) for v in results.vars]
    rows = [[str(val) if val is not None else None for val in row] for row in results]
    df = pd.DataFrame(rows, columns=cols)
    if title:
        print(f"\n{'='*60}")
        print(f"  {title}")
        print(f"  {len(df)} resultats")
        print(f"{'='*60}")
    return df

---
## 5. Resum de la pipeline

Estadístiques finals del graf construït.

In [14]:
# Comptar instàncies per classe
count_query = """
PREFIX wp: <http://webprotege.stanford.edu/>

SELECT ?classe (COUNT(?inst) AS ?num)
WHERE {
    ?inst a ?classe .
    FILTER (?classe IN (
        wp:RDHGr7pl4B25glJmq2xYYmN,   # Accidente
        wp:R7oZUszqj7aEjlsk698JmFM,   # AccidenteLeve
        wp:RYqhWnA3Yc3qHHEFBZcRoJ,   # AccidenteGrave
        wp:R8bzivJwhQYj45eHewQTg08,   # AccidenteMortal
        wp:RCyHtlgkOu2iks7ZQbQPeLw,   # AccidenteSinVictima
        wp:RCglr2LgVmFa85WTIfryLqR,   # Lugar
        wp:R8CyWdnPgD3YkoaC4jXyLH8,   # Persona
        wp:RBPXhVTcni9UaQtHolrwEGM,   # Conductor
        wp:RBjMNvYweizmQ8EE1HnWwNF,   # Vianant
        wp:RDfBEkaNQRJf9P95RTve7DV,   # EstacionMeteorologica
        wp:R8I54xpcV9jUWT2m34cWCxd    # LecturaMeteorologica
    ))
}
GROUP BY ?classe
ORDER BY DESC(?num)
"""

classe_map = {
    "http://webprotege.stanford.edu/RDHGr7pl4B25glJmq2xYYmN": "Accidente",
    "http://webprotege.stanford.edu/R7oZUszqj7aEjlsk698JmFM": "AccidenteLeve",
    "http://webprotege.stanford.edu/RYqhWnA3Yc3qHHEFBZcRoJ":  "AccidenteGrave",
    "http://webprotege.stanford.edu/R8bzivJwhQYj45eHewQTg08":  "AccidenteMortal",
    "http://webprotege.stanford.edu/RCyHtlgkOu2iks7ZQbQPeLw":  "AccidenteSinVictima",
    "http://webprotege.stanford.edu/RCglr2LgVmFa85WTIfryLqR":  "Lugar",
    "http://webprotege.stanford.edu/R8CyWdnPgD3YkoaC4jXyLH8":  "Persona",
    "http://webprotege.stanford.edu/RBPXhVTcni9UaQtHolrwEGM":  "Conductor",
    "http://webprotege.stanford.edu/RBjMNvYweizmQ8EE1HnWwNF":  "Vianant",
    "http://webprotege.stanford.edu/RDfBEkaNQRJf9P95RTve7DV":  "EstacioMeteo",
    "http://webprotege.stanford.edu/R8I54xpcV9jUWT2m34cWCxd":  "LecturaMeteo",
}

df_summary = run_sparql(count_query, "Resum — Instàncies per classe")
df_summary["classe"] = df_summary["classe"].map(classe_map).fillna(df_summary["classe"])
df_summary["num"] = df_summary["num"].astype(int)

print(f"\nTotal triples al graf: {len(g):,}")
print(f"Fitxer KG: {OUTPUT_KG} ({os.path.getsize(OUTPUT_KG)/1e6:.2f} MB)")
df_summary


  Resum — Instàncies per classe
  10 resultats

Total triples al graf: 257,593
Fitxer KG: kg_accidentes_bcn.ttl (15.71 MB)


,classe,num
0,LecturaMeteo,16396
1,Conductor,12953
2,Accidente,7741
3,AccidenteLeve,6599
4,Lugar,4386
5,Vianant,1039
6,AccidenteSinVictima,890
7,AccidenteGrave,241
8,AccidenteMortal,11
9,EstacioMeteo,3


---
## 7. Conclusions

Aquesta pipeline demostra com el model de graf permet:

1. **Joins multi-hop naturals** (Vianant → Accident → Lloc → Barri) sense JOIN explícits
2. **Herència semàntica** via la jerarquia de classes del TBOX (AccidenteMortal ⊆ Accidente)
3. **Queries declaratives** que expressen patrons estructurals del domini directament
4. **Integració de fonts** heterogènies (accidents + meteorologia) sota un vocabulari comú

El graf resultant pot ser consumit per la pipeline de KG embeddings (PyTorch/PyG) per tasques de ML.